In [ ]:
#Install libraries
!pip install pandas
!pip install numpy
!pip install scanpy

UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

--- Logging error ---
Call stack:
Traceback (most recent call last):
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/IPython/utils/_process_posix.py", line 125, in system
    child = pexpect.spawn(self.sh, args=['-c', cmd])  # Vanilla Pexpect
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/pexpect/pty_spawn.py", line 205, in __init__
    self._spawn(command, args, preexec_fn, dimensions)
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/pexpect/pty_spawn.py", line 303, in _spawn
    self.ptyproc = self._spawnpty(self.args, env=self.env,
                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/pexpect/pty_spawn.py", line 315, in _spawnpty
    return ptyprocess.PtyProcess.spawn(args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/ptyprocess/ptyp


During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/IPython/core/interactiveshell.py", line 3747, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "/var/folders/1j/bvnf9yzx3c38gdx8rsn3byc80000gn/T/ipykernel_28638/1416248573.py", line 1, in <module>
    get_ipython().system('pip install pandas')
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/ipykernel/zmqshell.py", line 788, in system_piped
    self.user_ns["_exit_code"] = system(self.var_expand(cmd, depth=1))
                                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/IPython/utils/_process_posix.py", line 141, in system
    child.sendline(chr(3))
    ^^^^^
UnboundLocalError: cannot access local variable 'child' where it is not associated with a value

During handling of the above exception, another excepti

In [6]:
#Import libraries
import pandas as pd
import numpy as np  
import scanpy as sc
import sys

In [7]:
#Load in dataset
df = pd.read_csv('/Users/LKaup/Documents/Uni/Master/Kurse/Master thesis/RNAseq/Raw_test_data/GSE233866_untreated_counts.csv', index_col=0)

# Convert to AnnData (cells as rows!)
adata = sc.AnnData(df.T)

In [8]:
#Annotate mitochondrial and ribosomal genes
# Mitochondrial genes (human: MT-, mouse: mt-)
adata.var['mt'] = adata.var_names.str.upper().str.startswith('MT-')

# Ribosomal genes (RPL, RPS)
adata.var['ribo'] = adata.var_names.str.upper().str.startswith(('RPL', 'RPS'))

In [9]:
#Compute QC metrics
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt', 'ribo'], inplace=True)

In [10]:
#Rename columns to match description
adata.obs['percent_mito'] = adata.obs['pct_counts_mt']
adata.obs['percent_ribo'] = adata.obs['pct_counts_ribo']

In [11]:
#Cell cycle scoring 
#Loads standard Seurat gene lists manually 
s_genes = [
    'Mcm6', 'Exo1', 'Dtl', 'Cdca7', 'Rad51', 'Wdr76', 'Pcna', 'Pola1', 'Ccne2', 'Casp8ap2', 'Usp1', 'Nasp', 'Clspn', 'Rpa2', 'Tyms', 'Slbp', 'Ung', 'Rfc2', 'Mcm2', 'Rad51ap1', 'E2f8', 'Blm', 'Pold3', 'Rrm1', 'Prim1', 'Mcm5', 'Gins2', 'Tipin', 'Brip1', 'Cdc6', 'Gmnn', 'Rrm2', 'Ubr7', 'Dscc1', 'Atad2', 'Mcm4', 'Cdc45', 'Chaf1b', 'Uhrf1', 'Msh2', 'Fen1', 'Hells'
]

g2m_genes = [
    'Hjurp', 'Nuf2', 'Lbr', 'Cenpf', 'Nek2', 'Tubb4b', 'Ckap5', 'Nusap1', 'Bub1', 'Ckap2l', 'Tpx2', 'Ube2c', 'Aurka', 'Ect2', 'Smc4', 'Cks1b', 'Anp32e', 'Psrc1', 'Cenpe', 'Cdc20', 'Cdca8', 'Cenpa', 'Tacc3', 'Cdca3', 'Mki67', 'Cdk1', 'Gas2l3', 'Tmpo', 'Ckap2', 'Hmgb2', 'Ctcf', 'Anln', 'Kif23', 'Ccnb2', 'Ttk', 'Hmmr', 'Aurkb', 'Top2a', 'Birc5', 'Cks2', 'G2e3', 'Gtse1', 'Cbx5', 'Ndc80', 'Kif20b', 'Kif11'
]

sc.tl.score_genes_cell_cycle(adata, s_genes=s_genes, g2m_genes=g2m_genes)

In [12]:
#Compute relative expression per gene per cell
#Normalize per cell (UMI fraction)
adata.layers["relative"] = adata.X / adata.X.sum(axis=1, keepdims=True)

In [13]:
#Check Malat1 expression
malat1_mask = adata.var_names.str.lower() == "malat1"

#Mean fraction across cells
malat1_fraction = np.mean(adata[:, malat1_mask].layers["relative"])

print("Mean Malat1 fraction:", malat1_fraction)

Mean Malat1 fraction: 0.04324125873767022


In [14]:
#Remove Malat1
adata = adata[:, ~(adata.var_names.str.lower() == "malat1")]

In [15]:
#Filter genes (present in ≥10 cells)
sc.pp.filter_genes(adata, min_cells=10)

/opt/anaconda3/envs/RNAseq/lib/python3.12/site-packages/scanpy/preprocessing/_simple.py:279: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  adata.var["n_cells"] = number


In [16]:
#Filter cells and libaries based on set thresholds
adata = adata[
    (adata.obs.n_genes_by_counts >= 500) &
    (adata.obs.n_genes_by_counts <= 10000) &
    (adata.obs.percent_mito < 5),
    :
]

In [17]:
#Filter for dopaminergic cells
#Gene names to match (case-insensitive)
genes = ["Th", "Slc6a3"]

#Check availability (optional sanity check)
available = adata.var_names.str.lower()
for g in genes:
    print(g, (available == g.lower()).any())

#Get expression matrices for each gene
th = adata[:, adata.var_names.str.lower() == "th"].X
slc6a3 = adata[:, adata.var_names.str.lower() == "slc6a3"].X

#Convert to boolean expression (> 0)
th_expr = np.array(th > 0).flatten()
slc6a3_expr = np.array(slc6a3 > 0).flatten()

#Keep cells expressing either gene (OR condition)
keep = th_expr | slc6a3_expr

#Subset AnnData object
adata_filtered = adata[keep, :].copy()

#Check result
print(adata_filtered.shape)

Th True
Slc6a3 True
(6002, 21180)


In [67]:
adata_filtered.write("QC_AnnDataObject.h5ad")